In [2]:
# %matplotlib widget
import threading
import time
import matplotlib.pyplot as plt
import nest_asyncio
import numpy as np
nest_asyncio.apply()  # required in notebooks

from phyling.ble import NanoPhyling, MiniPhyling

In [4]:
# Alarm example: play a beep when acc_x exceeds a threshold (uses NanoPhyling_19)
import numpy as np
import sounddevice as sd
import threading
import time

def make_beep(samplerate=22050, duration=0.18, freq=880.0, amp=0.3):
    t = np.linspace(0, duration, int(samplerate * duration), endpoint=False)
    tone = amp * np.sin(2 * np.pi * freq * t)
    fade = np.linspace(1.0, 0.0, tone.size)
    return (tone * fade).astype('float32'), samplerate

beep_buf, sr = make_beep()
last_play = {'t': 0.0}
threshold = 2.0  # m/s^2 (adjust as needed)
cooldown = 3.0    # seconds between alarms
axis = 'acc_x'

def _play_beep():
    sd.play(beep_buf, sr)
    sd.wait()

def alarm_callback(df):
    try:
        if axis not in df.columns:
            return
        vals = df[axis].abs()
        if (vals > threshold).any():
            now = time.time()
            if now - last_play['t'] < cooldown:
                return
            last_play['t'] = now
            threading.Thread(target=_play_beep, daemon=True).start()
    except Exception as e:
        print('Alarm callback error:', e)

# Create device with the sensor name you provided
device = NanoPhyling(ble_name="NanoPhyling_19")
device.on_data(alarm_callback)

# Run recording in background for 30 seconds (set None to run until interrupted)
t_alarm = threading.Thread(target=device.run, kwargs={'duration': 30}, daemon=True)
t_alarm.start()
print('Alarm monitoring started for NanoPhyling_19 (30s).')

Scanning for BLE sensor 'NanoPhyling_19'...Alarm monitoring started for NanoPhyling_19 (30s).



Found: NanoPhyling_19 (BA310A7D-B994-FCA2-F970-A9594C1A1922)
[NanoPhyling_19] Device NanoPhyling_19 Connected
[NanoPhyling_19] Recording started
[NanoPhyling_19] Recording stopped (5715 samples)
